In [ ]:
pip install transformers torch accelerate


In [ ]:
pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [2]:
from huggingface_hub import notebook_login

In [3]:
notebook_login()

In [4]:
model_name="meta-llama/Llama-3.2-1B-Instruct"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
result="""In the 16th century, an age of great marine and terrestrial exploration, Ferdinand Magellan led the first expedition to sail around the world. As a young Portuguese noble, he served the king of Portugal, but he became involved in the quagmire of political intrigue at court and lost the king’s favor. After he was dismissed from service by the king of Portugal, he offered to serve the future Emperor Charles V of Spain.

A papal decree of 1493 had assigned all land in the New World west of 50 degrees W longitude to Spain and all the land east of that line to Portugal. Magellan offered to prove that the East Indies fell under Spanish authority. On September 20, 1519, Magellan set sail from Spain with five ships. More than a year later, one of these ships was exploring the topography of South America in search of a water route across the continent. This ship sank, but the remaining four ships searched along the southern peninsula of South America. Finally they found the passage they sought near 50 degrees S latitude. Magellan named this passage the Strait of All Saints, but today it is known as the Strait of Magellan.

One ship deserted while in this passage and returned to Spain, so fewer sailors were privileged to gaze at that first panorama of the Pacific Ocean. Those who remained crossed the meridian now known as the International Date Line in the early spring of 1521 after 98 days on the Pacific Ocean. During those long days at sea, many of Magellan’s men died of starvation and disease.

Later, Magellan became involved in an insular conflict in the Philippines and was killed in a tribal battle. Only one ship and 17 sailors under the command of the Basque navigator Elcano survived to complete the westward journey to Spain and thus prove once and for all that the world is round, with no precipice at the edge."""

In [16]:
questions=["who was magellan", "Whom did Magellan serve","Who is Elcano","With how many ships did Magellan Sail from spain", "When did Magellan Sail from Spain", "How did Magellan Die", "How did Magellan's Men Die","Who is Sreekish Mooppil Sunil","What happened to the Ships that sailed from Spain"]

In [19]:
for question in questions:
  question_lower=question.lower().strip()
  is_person_question = (
    question_lower.startswith("who is ") or
    question_lower.startswith("who was "))
  if is_person_question:
    instruction="""You are a precise question answering system.
                Answer the question using ONLY information explicitly stated in the passage.
                For questions of the form "Who is X?" or "Who was X?":
                - Do NOT output X's name.
                - Return the COMPLETE description of X from the passage.
                - Include the occupation, role, title, nationality, and other descriptive words
                that belong to the description.
                - Do not return only one word from the description.
                Make the answer as short as possible."""
  else:
    instruction="""You are a precise question answering system.\n\n

            Answer the question using ONLY information explicitly stated
            in the provided passage.\n\n
            Return the COMPLETE answer phrase from the passage.
            Do not return only a keyword or a single word when the answer is part of a larger phrase.\n\n
            Write the answer as short as possible"""
  messages = [
    {
        "role": "system",
        "content": (
            instruction
        )
    },
    {
        "role": "user",
        "content": f"Text: {result}\n\nQuestion: {question}"
    }]
  inputs = tokenizer.apply_chat_template( messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
  inputs_ids= inputs["input_ids"].to(model.device)
  input_length = inputs_ids.shape[-1]
  outputs = model.generate(inputs_ids,max_new_tokens=100,do_sample=False)
  answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
  print(answer)

Ferdinand Magellan was a young Portuguese noble who served the king of Portugal, but became involved in the quagmire of political intrigue at court and lost the king's favor.
Portugal


KeyboardInterrupt: 